# Multimodal RAG with LlamaIndex

This notebook shows multimodal RAG on table, chart, and text extraction from a sample PDF using **LlamaIndex** and the **NeMo Retriever Library**.

**Note:** This notebook uses the **NeMo Retriever Library** (`nemo-retriever`) with local Hugging Face inference and **LanceDB** (default `./lancedb`, table `nv-ingest`). Install from the [NeMo-Retriever](https://github.com/NVIDIA/NeMo-Retriever) repository (`pip install "nemo-retriever[local]"` plus the CUDA 13 torch stack in [nemo_retriever/README.md](https://github.com/NVIDIA/NeMo-Retriever/blob/main/nemo_retriever/README.md)). Legacy NV-Ingest microservices, Docker Compose, `nv_ingest_client`, and Milvus are not part of the 26.05 customer path.

%pip install -qU llama_index llama-index-embeddings-nvidia llama-index-llms-nvidia "nemo-retriever[local]"

In [ ]:
Extract tables and charts from a sample PDF, embed them, and upload to LanceDB with `GraphIngestor`.

from pathlib import Path

from nemo_retriever.graph_ingestor import GraphIngestor
from nemo_retriever.params import EmbedParams, ExtractParams, VdbUploadParams

pdf = Path("../data/multimodal_test.pdf")
embed_params = EmbedParams(
    model_name="nvidia/llama-nemotron-embed-1b-v2",
    local_ingest_embed_backend="hf",
)

ingestor = (
    GraphIngestor(run_mode="inprocess", documents=[str(pdf.resolve())])
    .extract(
        ExtractParams(
            extract_text=False,
            extract_tables=True,
            extract_charts=True,
            extract_images=False,
        )
    )
    .embed(embed_params)
    .vdb_upload(
        VdbUploadParams(
            vdb_op="lancedb",
            uri="./lancedb",
            table_name="nv-ingest",
            overwrite=True,
        )
    )
)
ingestor.ingest()

In [ ]:
Retrieve context with `Retriever`, then use a LlamaIndex LLM to answer questions.

from nemo_retriever.retriever import Retriever

retriever = Retriever(
    lancedb_uri="lancedb",
    lancedb_table="nv-ingest",
    embedder="nvidia/llama-nemotron-embed-1b-v2",
    top_k=5,
    local_query_embed_backend="hf",
)

In [ ]:
Use [build.nvidia.com](https://build.nvidia.com/meta/llama-3_1-405b-instruct) for generation. Set `NVIDIA_API_KEY` before running the next cell.

import os

from llama_index.llms.nvidia import NVIDIA

os.environ["NVIDIA_API_KEY"] = "[YOUR NVIDIA API KEY HERE]"

llm = NVIDIA(model="meta/llama-3.1-405b-instruct")


def answer(question: str) -> str:
    hits = retriever.query(question)
    context = "\n\n".join(h.get("text", "") for h in hits if h.get("text"))
    prompt = (
        "Use the context below to answer the question.\n\n"
        f"Context:\n{context}\n\nQuestion: {question}\nAnswer:"
    )
    return str(llm.complete(prompt))

In [30]:
import os
from llama_index.llms.nvidia import NVIDIA

# TODO: Add your NVIDIA API key
os.environ["NVIDIA_API_KEY"] = "[YOUR NVIDIA API KEY HERE]"

llm = NVIDIA(model="meta/llama-3.1-405b-instruct")
query_engine = index.as_query_engine(llm=llm)

answer("What is the dog doing and where?")

In [9]:
query_engine.query("What is the dog doing and where?").response

'The dog is chasing a squirrel in the front yard.'